# Reproduce hIAPP screening

Run the cells in order to recompute scores and filters from the saved CSV.
Assertions check the reported counts, review tiers, and final candidates.
The outputs are model scores and sequence-property estimates, not experimental
measurements.


## 0. Imports and repository paths

The path lookup works both in JupyterLab and in automated notebook runs.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists():
    ROOT = ROOT.parent
assert (ROOT / "scripts/screen_candidates.py").exists(), ROOT
sys.path.insert(0, str(ROOT))

from scripts.screen_candidates import compute_scores, select

DATA = ROOT / "data/scored/merged_all_2_scored.csv"
FINAL = ROOT / "results/final_candidates/final_12.csv"
OUT = ROOT / "results/screening"
OUT.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Input:", DATA)

## 1. Load and audit the score table

The table contains the generated design/IF pool plus reference inhibitor rows.
Only `design` and `IF_02` enter candidate ranking.

In [ ]:
raw = pd.read_csv(DATA)
source_counts = raw["source"].value_counts().sort_index()
design_rows = int(raw["source"].isin(["design", "IF_02"]).sum())
display(source_counts.to_frame("rows"))
print("All rows:", len(raw))
print("Design/IF analysis rows:", design_rows)
assert len(raw) == 9884
assert design_rows == 9806

## 2. Recompute all direction-aware percentiles

Affinity consensus, head spread, structure quality, developability, the main
score and reference-space distance are recomputed from raw columns. No derived
Excel formulas are trusted.

In [ ]:
scored = compute_scores(raw)
assert len(scored) == 9806
view_cols = [
    "name", "d_sequence", "source", "structure_bottleneck",
    "affinity_consensus", "affinity_head_spread", "structure_quality",
    "developability_score", "overall_score",
]
display(scored[view_cols].sort_values("overall_score", ascending=False).head(10))

## 3. Apply the hard gate

Required: all six affinity fields, `min(pTM, ipTM) >= 0.65`, solubility >= 40
and aggregation risk <= 65.

In [ ]:
hard_pool, top32, p4_pool = select(scored)
print(f"Hard gate: {len(hard_pool):,} / {len(scored):,} ({len(hard_pool)/len(scored):.2%})")
assert len(hard_pool) == 904

funnel = pd.Series({
    "Scored designs": len(scored),
    "Hard gate": len(hard_pool),
    "P1-P3 review": len(top32),
    "P4 review": 10,
    "Final": 12,
})
display(funnel.to_frame("candidate_count"))

## 4. Reconstruct P1, P2 and P3

P1 emphasizes structure confidence and affinity-head agreement; P2 continues
ranking by the same main score; P3 preserves head disagreement or novelty.
Sequence similarity control is applied greedily in all three tiers.

In [ ]:
tier_counts = top32["priority"].value_counts().sort_index()
display(tier_counts.to_frame("count"))
assert tier_counts.to_dict() == {"P1": 12, "P2": 12, "P3": 8}

top32.to_csv(OUT / "p1_p2_p3_top32.csv", index=False)
display(top32[["priority", "name", "d_sequence", "overall_score",
               "structure_bottleneck", "affinity_consensus",
               "affinity_head_spread"]].head(12))

## 5. Reconstruct the finalized P4 long-peptide branch

The final P4 gate uses length 11-12 aa and `min(pTM, ipTM) >= 0.75`, plus the
same solubility/aggregation constraints. Thirty unique sequences pass; ten
proceed to structural review.

In [ ]:
print("P4 hard-pass unique sequences:", len(p4_pool))
assert len(p4_pool) == 30
p4_top10 = p4_pool.head(10).copy()
p4_top10.to_csv(OUT / "p4_top10.csv", index=False)
display(p4_top10[["name", "d_sequence", "length_aa", "p4_score",
                  "structure_bottleneck", "affinity_consensus",
                  "developability_score"]])

## 6. Verify the final 12 synthesis manifest

The final choice followed manual complex-structure review. This cell verifies
that every selected sequence belongs to the documented review pool and that the
tier composition is P1 × 8, P2 × 1, P3 × 1, and P4 × 2.

In [ ]:
final = pd.read_csv(FINAL)
assert len(final) == 12
assert final["d_peptide_sequence"].is_unique
assert final["wetlab_activity"].eq("not_measured").all()
assert final["priority"].value_counts().to_dict() == {"P1": 8, "P4": 2, "P2": 1, "P3": 1}

main_review = set(top32["d_sequence"])
p4_review = set(p4_top10["d_sequence"])
for row in final.itertuples():
    expected = p4_review if row.priority == "P4" else main_review
    assert row.d_peptide_sequence in expected, row
    assert (ROOT / "results/final_candidates/structures" / row.structure_file).is_file()
display(final)

## 7. Export audit tables and summary

These audit tables are regenerated by the workflow. The committed final
manifest remains the authoritative manual-review record.

In [ ]:
hard_pool.to_csv(OUT / "hard_gate_904.csv", index=False)
summary = pd.DataFrame([
    ["analysis_design_rows", len(scored)],
    ["hard_gate", len(hard_pool)],
    ["P1", int((top32.priority == "P1").sum())],
    ["P2", int((top32.priority == "P2").sum())],
    ["P3", int((top32.priority == "P3").sum())],
    ["P4_hard_pass", len(p4_pool)],
    ["P4_review", len(p4_top10)],
    ["final_selected", len(final)],
], columns=["stage", "count"])
summary.to_csv(OUT / "screening_summary.csv", index=False)
display(summary)
print("Reproduction checks passed.")

## Next experimental step

Add immutable ThT time-series measurements under `wetlab/raw/`, include both
positive controls and appropriate negative/vehicle controls, retain independent
and technical repeats, and analyze concentration response with uncertainty.
Until those data exist, the correct conclusion is that these 12 sequences are
**computationally prioritized candidates selected for synthesis**.